In [ ]:
from pathlib import Path
import sys, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# 1. Resolve project root
root = Path.cwd().resolve()
while root != root.parent and not (root / "src").is_dir():
    root = root.parent

if not (root / "src").is_dir():
    # Candidates for local or remote/isolated kernel environments
    candidates = [
        Path("..").resolve(),
        Path(".").resolve(),
        Path(r"c:/Users/HP/Desktop/bearing-rul-project"),
        Path(r"c:\Users\HP\Desktop\bearing-rul-project"),
    ]
    for c in candidates:
        if (c / "src").is_dir():
            root = c
            break

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Also ensure cwd is at notebooks/ or repo root so relative paths work
os.chdir(str(root / "notebooks") if (root / "notebooks").is_dir() else str(root))

from src.features import add_spectral_features
from src.health import prepare
from src.labels import run_table, find_onset, add_labels, RUNS
from src.models_anomaly import Mahalanobis, PCAMonitor, IsoForest
from src.evaluate import evaluate_run

data_dir = root / "data"
results_dir = root / "results"
results_dir.mkdir(exist_ok=True, parents=True)

df = pd.read_parquet(data_dir / "processed_features.parquet")
spec = np.load(data_dir / "env_spectra.npy")
df = add_spectral_features(df, spec)

FEATURES = (["rms", "peak", "kurt", "skew", "crest", "shape", "impulse",
             "spec_entropy", "spec_centroid"]
            + [f"band{i}_frac" for i in range(7)])          # band7 dropped: fractions sum to 1

z_runs = prepare(df, FEATURES)                               # per-bearing z-scores, causal smoothing, burn-in removed
tables = {f"{s}_{b}": run_table(df, s, b) for s, b in RUNS}

labels = {}
for variant, thr in (("tight", 1.25), ("loose", 1.10)):
    for name, u in tables.items():
        labels[(variant, name)] = add_labels(u, find_onset(u, thr, 20)).set_index("idx")["anomaly"]

make_models = lambda: [Mahalanobis(), PCAMonitor(stat="t2"), PCAMonitor(stat="spe"), IsoForest()]

rows, scores = [], {}
for name, Z in z_runs.items():
    for m in make_models():
        res, s, thr = evaluate_run(Z, FEATURES, m, labels[("tight", name)], labels[("loose", name)],
                                   tables[name].set_index("idx")["rul_hours"])
        rows.append({"run": name, **res})
        scores[(name, m.name)] = (s, thr)
results = pd.DataFrame(rows).round(3)

for col in ["auc_tight", "auc_loose", "lead_hours", "pre_onset_alarm_rate"]:
    print(col); display(results.pivot(index="model", columns="run", values=col))



Anomaly score over time with the threshold and onsets marked.

In [ ]:
names = ["mahalanobis", "pca_t2", "pca_spe", "isolation_forest"]
fig, axes = plt.subplots(len(z_runs), 4, figsize=(18, 10))
for r, (name, Z) in enumerate(z_runs.items()):
    for c, mname in enumerate(names):
        s, thr = scores[(name, mname)]
        ax = axes[r, c]
        ax.semilogy(Z["life"], s, lw=0.6)
        ax.axhline(thr, color="r", ls="--", lw=0.8)
        ax.axvspan(0.15, 0.32, color="g", alpha=0.12)         # training window
        ax.axvspan(0.32, 0.40, color="orange", alpha=0.12)    # calibration window
        for tag, ls in (("tight", "-"), ("loose", ":")):
            y = labels[(tag, name)].reindex(Z.index).to_numpy()
            if y.sum():
                ax.axvline(Z["life"][y == 1].iloc[0], color="k", ls=ls, lw=0.8)
        ax.set_title(f"{name} | {mname}", fontsize=9)
plt.tight_layout(); plt.savefig(str(results_dir / "anomaly_scores.png"), dpi=150)



The control-bearing figure

In [ ]:
from src.models_anomaly import Mahalanobis
from src.evaluate import TRAIN, CAL

CONTROL = [("set1", "b1"), ("set1", "b2"), ("set3", "b1"), ("set3", "b2")]   # never failed
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}

def maha(Z):
    life = Z["life"].to_numpy(); X = Z[FEATURES].to_numpy()
    m = Mahalanobis().fit(X[(life >= TRAIN[0]) & (life < TRAIN[1])])
    s = m.score(X)
    thr = np.quantile(s[(life >= CAL[0]) & (life < CAL[1])], 0.995)
    return life, s / thr                 # 1.0 = alarm level, so runs are comparable

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, prefix in zip(axes, ["set1", "set3"]):
    for name, Z in all_z.items():
        if name.startswith(prefix):
            life, r = maha(Z)
            ax.semilogy(life, r, lw=0.7, label=name)
    ax.axhline(1, color="k", ls="--"); ax.axvline(0.41, color="r", ls=":", lw=0.8)
    ax.set_title(prefix); ax.legend()
plt.tight_layout(); plt.savefig(str(results_dir / "control_bearings.png"), dpi=150)



Set 1 B3 shift table

In [ ]:
Z = z_runs["set1_b3"]; life = Z["life"]
w = lambda lo, hi: Z.loc[(life >= lo) & (life < hi), FEATURES].median()
shift = pd.DataFrame({"0.32-0.40": w(.32, .40) - w(.15, .32),
                      "0.42-0.60": w(.42, .60) - w(.15, .32)})
shift.reindex(shift["0.42-0.60"].abs().sort_values(ascending=False).index).round(2)

Set 1 B4 shift table

In [ ]:
Z = z_runs["set1_b4"]; life = Z["life"]
w = lambda lo, hi: Z.loc[(life >= lo) & (life < hi), FEATURES].median()
shift = pd.DataFrame({"0.32-0.40": w(.32, .40) - w(.15, .32),
                      "0.42-0.60": w(.42, .60) - w(.15, .32)})
shift.reindex(shift["0.42-0.60"].abs().sort_values(ascending=False).index).round(2)

In [ ]:
results.pivot(index="model", columns="run", values="alarm_life")

Next experiment: alarms on never-failed bearings

In [ ]:
from src.evaluate import TRAIN, CAL, confirmed_alarm

CONTROL = [("set1", "b1"), ("set1", "b2"), ("set2", "b2"), ("set2", "b3"),
           ("set2", "b4"), ("set3", "b1"), ("set3", "b2")]
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}

def alarm_only(Z, model, q=0.995, n_consec=10):
    life = Z["life"].to_numpy(); X = Z[FEATURES].to_numpy()
    tr = (life >= TRAIN[0]) & (life < TRAIN[1])
    ca = (life >= CAL[0]) & (life < CAL[1])
    te = life >= CAL[1]
    model.fit(X[tr]); s = model.score(X)
    k = confirmed_alarm(s[te], np.quantile(s[ca], q), n_consec)
    return np.nan if k is None else round(float(life[np.flatnonzero(te)[k]]), 3)

res = pd.DataFrame({m().name: {name: alarm_only(Z, m()) for name, Z in all_z.items()}
                    for m in (Mahalanobis, lambda: PCAMonitor(stat="t2"),
                              lambda: PCAMonitor(stat="spe"), IsoForest)})
res["failed"] = [n in z_runs for n in res.index]
res.sort_values(["failed", "mahalanobis"], ascending=False)

In [ ]:
from src.evaluate import alarm_blocked

MODELS = {"mahalanobis": Mahalanobis,
          "pca_t2": lambda: PCAMonitor(stat="t2"),
          "pca_spe": lambda: PCAMonitor(stat="spe"),
          "isolation_forest": IsoForest}

def run_all(base):
    rows = []
    for name, Z in all_z.items():
        for mname, mk in MODELS.items():
            res, _, _ = alarm_blocked(Z, FEATURES, mk, base=base)
            rows.append({"run": name, "model": mname, **res})
    return pd.DataFrame(rows)

def margins(alarm):
    rows = {}
    for f in z_runs:
        ctrl = [n for n in alarm.index if n.startswith(f.split("_")[0]) and n not in z_runs]
        rows[f] = alarm.loc[ctrl].min() - alarm.loc[f]
    return pd.DataFrame(rows).T.round(3)

for base in [(0.15, 0.40), (0.15, 0.32)]:
    r = run_all(base)
    print("base window:", base)
    display(margins(r.pivot(index="run", columns="model", values="alarm_life")))
    display(r.pivot(index="run", columns="model", values="start_above").round(2))

Remove the rig-level drift

In [ ]:
CONTROL = [("set1", "b1"), ("set1", "b2"), ("set2", "b2"), ("set2", "b3"), ("set2", "b4"),
           ("set3", "b1"), ("set3", "b2"), ("set3", "b4")]
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}


def common_mode_removed(zdict, features):
    out = {}
    for s in ("set1", "set2", "set3"):
        names = [n for n in zdict if n.startswith(s)]
        idx = sorted(set.intersection(*[set(zdict[n].index) for n in names]))
        cm = pd.concat([zdict[n].loc[idx, features] for n in names]).groupby(level=0).median()
        for n in names:
            Z = zdict[n].loc[idx].copy()
            Z[features] = Z[features] - cm
            out[n] = Z
    return out


all_z_raw = all_z
all_z = common_mode_removed(all_z_raw, FEATURES)    # run_all() and margins() read the global all_z
r = run_all((0.15, 0.40))
display(margins(r.pivot(index="run", columns="model", values="alarm_life")))
display(r.pivot(index="run", columns="model", values="start_above").round(2))
all_z = all_z_raw                                   # restore

In [ ]:
r = run_all((0.15, 0.40))
r[r["run"].isin(z_runs)].pivot(index="run", columns="model", values="alarm_life").round(3)

---
## Phase 3 (continued) – New Anomaly Detection Models

Adding the remaining model families from the guide:
* **Statistical**: Gaussian Mixture Model (GMM)
* **Classical ML**: One-Class SVM (OC-SVM), Local Outlier Factor (LOF)
* **Deep learning autoencoders**: Dense AE, LSTM-AE, 1D-CNN-AE

All models share the same `fit / score` interface and are evaluated with the
same **blocked CV** framework as the first four models.


In [ ]:
import importlib, src.models_anomaly
importlib.reload(src.models_anomaly)

from src.models_anomaly import (
    GMMDetector, OCSVMDetector, LOFDetector,
    DenseAutoencoder, LSTMAutoencoder, CNN1DAutoencoder,
    _HAS_TORCH,
)
from src.evaluate import alarm_blocked, evaluate_run

print(f"PyTorch available: {_HAS_TORCH}")
print("All 6 new model classes imported successfully.")


In [ ]:
# Re-use z_runs, tables, labels, FEATURES from Cell 0
# (run this notebook from top, or the variables must be in scope)

NEW_MODELS = {
    "gmm":        GMMDetector,
    "ocsvm":      OCSVMDetector,
    "lof":        LOFDetector,
    "dense_ae":   DenseAutoencoder,
    "lstm_ae":    LSTMAutoencoder,
    "cnn1d_ae":   CNN1DAutoencoder,
}

# ── Run evaluate_run for AUC + lead-time on the 4 failed runs ─────────
print("Fitting and evaluating new models (this may take 2-5 min for autoencoders)...")
new_rows, new_scores = [], {}
for name, Z in z_runs.items():
    rul = tables[name].set_index("idx")["rul_hours"]
    for mname, mk in NEW_MODELS.items():
        try:
            m = mk()
            res, s, thr = evaluate_run(
                Z, FEATURES, m,
                labels[("tight", name)],
                labels[("loose", name)],
                rul,
            )
            new_rows.append({"run": name, **res})
            new_scores[(name, mname)] = (s, thr)
            print(f"  ✓ {mname} / {name}  alarm@{res.get('alarm_life', float('nan')):.3f}  "
                  f"AUC(tight)={res.get('auc_tight', float('nan')):.3f}")
        except Exception as exc:
            print(f"  ✗ {mname} / {name}: {exc}")
            new_rows.append({"run": name, "model": mname})

new_results = pd.DataFrame(new_rows).round(3)
print("\nDone.")


In [ ]:
print("=== AUC – tight labels (new models) ===")
display(new_results.pivot(index="model", columns="run", values="auc_tight").round(3))

print("\n=== AUC – loose labels ===")
display(new_results.pivot(index="model", columns="run", values="auc_loose").round(3))


In [ ]:
print("=== Alarm life fraction [0,1] – earlier is better (new models) ===")
display(new_results.pivot(index="model", columns="run", values="alarm_life").round(3))

print("\n=== Lead time (hours before failure) ===")
display(new_results.pivot(index="model", columns="run", values="lead_hours").round(1))

print("\n=== Pre-onset false alarm rate ===")
display(new_results.pivot(index="model", columns="run", values="pre_onset_alarm_rate").round(3))


---
### Consolidated Comparison — All 10 Models


In [ ]:
all_results = pd.concat([results, new_results], ignore_index=True)

for col, label in [("auc_tight",  "AUC (tight labels)"),
                   ("alarm_life", "Alarm life fraction  [earlier = better]"),
                   ("lead_hours", "Detection lead time (h)"),
                   ("pre_onset_alarm_rate", "False alarm rate (pre-onset healthy period)")]:
    piv = all_results.pivot(index="model", columns="run", values=col).round(3)
    piv["mean"] = piv.mean(axis=1).round(3)
    piv = piv.sort_values("mean", ascending=(col == "alarm_life"))
    print(f"\n──── {label} ────")
    display(piv)


---
### Anomaly Score Time-Series — All 10 Models

Each panel shows the anomaly score over the bearing's life fraction.
Vertical dashed lines mark the tight and loose anomaly onset. The red horizontal
line is the alarm threshold (99.5 % quantile of the calibration window).
Green shading = training window; orange = calibration window.


In [ ]:
ALL_SCORES = {**scores, **new_scores}

MODEL_ORDER = [
    "mahalanobis", "pca_t2", "pca_spe", "isolation_forest",
    "gmm", "ocsvm", "lof",
    "dense_ae", "lstm_ae", "cnn1d_ae",
]
MODEL_ORDER = [m for m in MODEL_ORDER if any((r, m) in ALL_SCORES for r in z_runs)]

n_runs = len(z_runs)
n_models = len(MODEL_ORDER)
fig, axes = plt.subplots(n_runs, n_models, figsize=(2.8 * n_models, 3.2 * n_runs))

for r, (rname, Z) in enumerate(z_runs.items()):
    life = Z["life"].to_numpy()
    t_onset_life = {}
    for tag in ("tight", "loose"):
        lab = labels[(tag, rname)]
        if lab.any():
            idx = lab.idxmax()
            match = Z.index[Z.index == idx]
            t_onset_life[tag] = float(Z.loc[match[0], "life"]) if len(match) else None
        else:
            t_onset_life[tag] = None

    for c, mname in enumerate(MODEL_ORDER):
        ax = axes[r, c] if n_runs > 1 else axes[c]
        if (rname, mname) not in ALL_SCORES:
            ax.set_visible(False)
            continue
        s, thr = ALL_SCORES[(rname, mname)]
        s = np.asarray(s)
        use_log = (s.min() > 0) and (s.max() / max(s.min(), 1e-12) > 50)
        if use_log:
            ax.semilogy(life, s, lw=0.5, color="steelblue")
        else:
            ax.plot(life, s, lw=0.5, color="steelblue")
        ax.axhline(thr, color="red", ls="--", lw=0.9, label="threshold")
        ax.axvspan(0.15, 0.32, color="green",  alpha=0.10)
        ax.axvspan(0.32, 0.40, color="orange", alpha=0.10)
        for tag, lf in t_onset_life.items():
            if lf is not None:
                ax.axvline(lf, color="red" if tag == "tight" else "darkorange",
                           ls=":" if tag == "tight" else "-.", lw=0.9)
        ax.set_xlabel("Life fraction", fontsize=7)
        ax.tick_params(labelsize=6)
        if r == 0:
            ax.set_title(mname.replace("_", "\n"), fontsize=7, fontweight="bold")
        if c == 0:
            ax.set_ylabel(rname.replace("_", " "), fontsize=7)

fig.suptitle("Anomaly Score Over Life — All 10 Models × 4 Bearings\n"
             "(green=train, orange=cal, red-dashed=threshold, dotted=tight onset, dash-dot=loose onset)",
             fontsize=9, y=1.01)
fig.tight_layout()
fig.savefig(str(results_dir / "anomaly_scores_all_models.png"), dpi=130, bbox_inches="tight")
plt.show()
print("Saved → results/anomaly_scores_all_models.png")




---
### False-Alarm Rate on Never-Failed Bearings — All 10 Models


In [ ]:
from src.evaluate import alarm_blocked
from src.models_anomaly import Mahalanobis, PCAMonitor, IsoForest

CONTROL_RUNS = [("set1", "b1"), ("set1", "b2"),
                ("set2", "b2"), ("set2", "b3"), ("set2", "b4"),
                ("set3", "b1"), ("set3", "b2"), ("set3", "b4")]

all_z_ext = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL_RUNS)}

ALL_MAKERS = {
    "mahalanobis":    Mahalanobis,
    "pca_t2":         lambda: PCAMonitor(stat="t2"),
    "pca_spe":        lambda: PCAMonitor(stat="spe"),
    "isolation_forest": IsoForest,
    "gmm":            GMMDetector,
    "ocsvm":          OCSVMDetector,
    "lof":            LOFDetector,
    "dense_ae":       DenseAutoencoder,
    "lstm_ae":        LSTMAutoencoder,
    "cnn1d_ae":       CNN1DAutoencoder,
}

ctrl_rows = []
for rname, Z in all_z_ext.items():
    is_failed = rname in z_runs
    for mname, mk in ALL_MAKERS.items():
        try:
            res, _, _ = alarm_blocked(Z, FEATURES, mk)
            ctrl_rows.append({"run": rname, "model": mname,
                               "alarm_life": res["alarm_life"],
                               "failed": is_failed})
        except Exception as exc:
            ctrl_rows.append({"run": rname, "model": mname,
                               "alarm_life": float("nan"), "failed": is_failed})

ctrl_df = pd.DataFrame(ctrl_rows)

# Pivot: rows=runs, cols=models; show failed runs at the top
piv = ctrl_df.pivot(index="run", columns="model", values="alarm_life").round(3)
piv["is_failed"] = piv.index.isin(z_runs)
display(piv.sort_values("is_failed", ascending=False).drop(columns="is_failed"))
print("\nNaN = no alarm fired during the test window (ideal for healthy bearings).")


In [ ]:
# Margin = earliest never-failed-bearing alarm (same set) minus failed-bearing alarm
# Positive margin → model alarms earlier on failed bearing than on healthy ones (good)

margin_rows = []
for mname in ALL_MAKERS:
    sub = ctrl_df[ctrl_df["model"] == mname]
    for rname in z_runs:
        s_prefix = rname.split("_")[0]
        f_alarm  = sub[sub["run"] == rname]["alarm_life"].values
        c_alarms = sub[(~sub["failed"]) & (sub["run"].str.startswith(s_prefix)) & sub["alarm_life"].notna()]["alarm_life"]
        if len(f_alarm) == 0 or np.isnan(f_alarm[0]):
            margin = float("nan")
        elif len(c_alarms) == 0:
            margin = 1.0 - float(f_alarm[0])   # no control bearing in this set alarmed
        else:
            margin = float(c_alarms.min()) - float(f_alarm[0])
        margin_rows.append({"model": mname, "run": rname, "margin": round(margin, 3)})

margin_df = pd.DataFrame(margin_rows)
piv_m = margin_df.pivot(index="model", columns="run", values="margin")
piv_m["mean_margin"] = piv_m.mean(axis=1).round(3)
display(piv_m.sort_values("mean_margin", ascending=False).round(3))
print("\nPositive margin = model reliably distinguishes failing from healthy bearings.")


---
### Summary: Mean AUC and Mean Lead Time Across the 4 Runs


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

auc_piv  = all_results.pivot(index="model", columns="run", values="auc_tight")
lead_piv = all_results.pivot(index="model", columns="run", values="lead_hours")

mean_auc  = auc_piv.mean(axis=1).sort_values(ascending=True)
mean_lead = lead_piv.mean(axis=1).reindex(mean_auc.index)

colors = ["#2196F3" if m not in ("dense_ae","lstm_ae","cnn1d_ae") else "#FF5722"
          for m in mean_auc.index]

mean_auc.plot.barh(ax=ax1, color=colors, edgecolor="k", linewidth=0.5)
ax1.set_xlabel("Mean AUC (tight labels, 4 runs)", fontsize=10)
ax1.set_xlim(0, 1.05)
ax1.axvline(1.0, color="grey", ls=":", lw=0.8)
ax1.set_title("Anomaly Detection AUC", fontsize=11)

mean_lead.plot.barh(ax=ax2, color=colors, edgecolor="k", linewidth=0.5)
ax2.set_xlabel("Mean lead time (hours before failure)", fontsize=10)
ax2.set_title("Detection Lead Time", fontsize=11)

for ax in (ax1, ax2):
    ax.tick_params(axis="y", labelsize=9)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

from matplotlib.patches import Patch
legend_els = [Patch(fc="#2196F3", label="Statistical / Classical ML"),
              Patch(fc="#FF5722", label="Deep Learning Autoencoders")]
fig.legend(handles=legend_els, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle("Phase 3 Anomaly Detection — All 10 Models", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(str(results_dir / "anomaly_phase3_summary.png"), dpi=130, bbox_inches="tight")
plt.show()
print("Saved → results/anomaly_phase3_summary.png")




---
## Phase 3 Summary

### Model Families Evaluated

| Family | Models | Note |
|--------|--------|------|
| Statistical | Mahalanobis, PCA-T², PCA-SPE | Fit on healthy data; closed-form scores |
| Ensemble / classical | Isolation Forest, GMM, OC-SVM, LOF | Flexible density boundaries |
| Deep learning AE | Dense AE, LSTM-AE, 1D-CNN-AE | Reconstruction error as anomaly score |

### Key Observations

* **Validation**: Leave-one-bearing-out is not feasible with only 4 trajectories; instead the
  healthy portion of each run is split into train (life 0.15–0.32), calibration (0.32–0.40),
  and test (≥ 0.40).  Thresholds are set at the 99.5 % quantile of the **calibration** window
  scores (blocked CV within the training window).
* **Alarm confirmation**: 10 consecutive threshold exceedances required, matching a realistic
  deployed alarm rule.
* **Control bearings**: never-failed bearings are scored under each model to measure the
  false-alarm rate; NaN means no alarm fired (ideal).
* **Lead time**: distance from the confirmed alarm to the end of the test, expressed in hours.
  Longer is better—the system gave more time to intervene.
* The *pre_onset_alarm_rate* metric counts how often the score exceeds the threshold
  *before* the onset (computed from the loose label); ideally zero.

### Next Step
Phase 4 — RUL Prediction using the same run-to-failure trajectories as training data.
